<div dir="rtl"><h2>03 — מצב לפני שריקה: פעימה A</h2>
<p>לפני שחוזים משחק, צריך לדעת איך נראתה הליגה בבוקר שלו. <b>M0</b> בונה לכל קבוצה שורה מתוך תוצאות בתאריכים קודמים ממש, לפי שעון ישראל. משחקים מאותו יום אינם רואים זה את זה.</p>
<p><b>טבלה על המגרש, בלי הורדות; עשויה להיות שונה מהטבלה הרשמית.</b> נקודות הן שלוש לניצחון ואחת לתיקו; בשתי עונות החצייה מעגלים למעלה. 11 המשחקים הטכניים אינם קלט לפיצ׳רים.</p>
<p>בפעימה זו: חוקי ליגה, טבלת as_of, רישום הפיצ׳רים ודגל מתמטי שמרני. Elo, כושר, התקפה/הגנה והקשר מהעונה הקודמת יגיעו בפעימות B ו-C. הסטטיסטיקות כאן על development בלבד: <bdi dir="ltr">2006/2007–2023/2024</bdi>. עונות ההחזקה מופיעות רק בשער האימות ההנדסי המאושר.</p>
<ol><li>הגדרות וקלט</li><li>חוקים ואימות</li><li>הטבלה בבוקר המשחק</li><li>רשמי מול ספורטיבי</li><li>יעדים פתוחים ושלב העונה</li><li>שמירה והמשך</li></ol></div>

<div dir="rtl"><h2>סביבת העבודה</h2><p><b>מה התא עושה:</b> טוען את פונקציות החבילה, קובע את שורש הפרויקט ומוודא שימוש ב-venv. הטבלאות מוצגות בעברית וב-RTL.</p></div>

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import HTML, display

from ligat.analysis.prematch_review import (
    championship_examples,
    dead_game_summary,
    postponed_example,
    registry_overview,
    rules_overview,
    split_comparison,
    sporting_official_comparison,
    stage_end_distribution,
    validation_overview,
)
from ligat.analysis.prematch_validation import load_technical_ledger, validate_prematch_tables
from ligat.analysis.prepare import DEVELOPMENT_SEASONS, load_analysis_tables
from ligat.data.repository import Repository
from ligat.features.artifacts import input_fingerprint, save_table_state
from ligat.features.registry import validate_feature_columns
from ligat.features.table_state import TableStateBlock
from ligat.simulate.rules import load_season_rules
from ligat.viz.prematch import dead_games_chart, review_table, stage_end_chart

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Select the project venv kernel"
started = time.perf_counter()
pio.renderers.default = "plotly_mimetype"
display(HTML('<div dir="rtl">סביבת ה-venv אומתה; החישובים מיובאים מהחבילה.</div>'))

<div dir="rtl"><h2>טעינת הקלט לפיצ׳רים</h2><p><b>מה התא עושה:</b> טוען עונות פיתוח דרך load_analysis_tables בלבד. הפיצ׳רים ייבנו ב-DataFrame חדש; הטבלאות הנקיות אינן משתנות. עמודות EDA קיימות, סיכומי סיום ושדות שנדחו אינם נכנסים לחישוב.</p></div>

In [ ]:
tables = load_analysis_tables(seasons=DEVELOPMENT_SEASONS)
rules = load_season_rules()
display(
    HTML(
        f'<div dir="rtl">נטענו {len(tables.match_table):,} משחקים '
        f"ו-{len(tables.team_matches):,} שורות קבוצה×משחק בעונות הפיתוח.</div>"
    )
)

<div dir="rtl"><h2>1. חוקי הליגה וקווי היעד</h2><p><b>מה התא עושה:</b> מציג את הפורמט, החצייה, שוברי השוויון וקווי העליון/הירידה לפי עונה. בפלייאוף מוצגים גודל הקבוצה ומספר המשחקים לכל קבוצה.</p>
<p>ב-<bdi dir="ltr">2011/2012</bdi> ירדו שלוש קבוצות. בשתי העונות שקדמו לה מקום 14 הוביל למבחן ונחשב כאן חלק מאזור הסיכון. מפגש ישיר נשאר שובר שוויון בתקנון; שוויון מוחלט מקבל מיקום ממוצע. בפיצול משתמשים במיקום העונה הקודמת אם עדיין נותר שוויון.</p></div>

In [ ]:
display(review_table(rules_overview(rules, list(DEVELOPMENT_SEASONS))))

<div dir="rtl"><h2>2. שער האימות — טבלאות רשמיות כבדיקה בלבד</h2><p><b>מה התא עושה:</b> משווה 9,956 שורות מחזור בעונות <bdi dir="ltr">2006/2007–2025/2026</bdi>, וכן טבלאות סיום. זו בדיקת הנדסה; אין מחקר או בחירת פיצ׳רים על עונות ההחזקה.</p>
<p>מונים נבדקים במחזורים ללא משחק שחוצה מחזורים. המשחקים הטכניים משוחזרים בספר נפרד לצורך הסבר הפער בלבד. מיקום רשמי נבדק ללא הורדות. בקצות השלבים האימות קפדני; באמצע העונה מותר סדר תצוגה אחר רק בתוך קבוצות שוות בנקודות ובהפרש, עם סף עצירה של 1%.</p>
<p>ארטיפקט מקור מאומת ב-<bdi dir="ltr">2006/2007</bdi>: בטבלה הרשמית חסרים שערי חלק ממשחקי התיקו. מותר רק פער שווה בשערי הזכות והחובה; נקודות, נ/ת/ה והפרש נשארים מדויקים. התוצאות עצמן אומתו מול אירועי השערים. שום כלל דירוג אינו משתנה בגלל אופן הצגת הטבלה באתר.</p><p><b>תוצאות האימות:</b> במחזורים הנקיים 7,572 מתוך 8,196 שורות תואמות ישירות (92.39%). יתר השורות מוסברות ב-467 פערים טכניים וב-157 ארטיפקטים; כולן עברו לפי המדיניות שאושרה. 1,760 שורות של מחזורים חוצים הוחרגו. נקודות הסיום עברו בכל 280 השורות אחרי הסבר הטכניים. סדר תצוגה: 12 מתוך 9,432 שורות ביניים (0.13%), מתחת לסף 1%.</p></div>

In [ ]:
engineering_tables = load_analysis_tables(seasons=list(range(8, 28)))
repo = Repository()
official_rounds = repo.read_filtered("round_tables", [("season_id", "in", list(range(8, 28)))])
official_finals = repo.read_filtered("final_tables", [("season_id", "in", list(range(8, 28)))])
validation = validate_prematch_tables(
    engineering_tables,
    rules,
    official_rounds,
    official_finals,
    load_technical_ledger(list(range(8, 28))),
)
assert len(official_rounds) == 9956
display(review_table(validation_overview(validation)))
display(
    HTML(
        '<div dir="rtl"><b>שער האימות עבר.</b> ״זהה בספורטיבי״ מציין התאמה ישירה; '
        "פער טכני וארטיפקט מקור הם פערים מוסברים שנבדקו, "
        "ומחזורים חוצים אינם נכללים באחוז ההתאמה.</div>"
    )
)

<div dir="rtl"><h2>3. בניית הטבלה בבוקר המשחק</h2><p><b>מה התא עושה:</b> יוצר שתי שורות לכל משחק, עם מפתחות game_id ו-team_id. כל שורה משתמשת רק במשחקים מתאריך קודם באותה עונה. הפערים חתומים: נקודות קו היעד פחות נקודות הקבוצה; ערך חיובי משמעו שהקבוצה מפגרת בנקודות אחרי הקו.</p>
<p>בפלייאוף, המיקום הכללי הוא היסט הקבוצה בטבלה ועוד המיקום בתוכה, גם אם לקבוצה בתחתון יותר נקודות מקבוצה בעליון. לכן פער בנקודות בין קבוצות פלייאוף אינו אפשרות לעבור ביניהן. אחרי הפיצול, פער לקו שאינו נגיש הוא NA: פער לירידה בעליון ובאמצעי, ופער למובילה באמצעי ובתחתון; פער לקו העליון קיים רק בסדיר לפני פיצול. נקודות למשחק הן הנקודות המחושבות, כולל חצייה כשכבר חלה, חלקי המשחקים ששוחקו; לפני משחק ראשון הערך הוא אפס.</p></div>

In [ ]:
build_started = time.perf_counter()
features = TableStateBlock(rules).compute(tables)
build_seconds = time.perf_counter() - build_started
validate_feature_columns(features, "table_state")
assert len(features) == 2 * len(tables.match_table)
display(
    HTML(
        f'<div dir="rtl">נבנו {len(features):,} שורות ו-{len(features.columns)} עמודות '
        f"ב-{build_seconds:.2f} שניות. הקלט והטבלאות הרשמיות אינם משתנים.</div>"
    )
)

<div dir="rtl"><h2>משחק נדחה: מספר המחזור אינו ציר זמן</h2><p><b>השאלה:</b> למה לא לקחת את הטבלה הרשמית של המחזור?</p><p><b>מה התא עושה:</b> מוצא משחק אמיתי של קבוצה שמשחק ממחזור קודם שלה הושלם רק אחריו, ומשווה בין שורת הבוקר שלה לטבלה הרשמית.</p>
<p>הטבלה הרשמית היא אחרי המחזור, ולכן היא כוללת גם את תוצאת המשחק הנחזה ומשחקים נוספים מאותו מחזור. הדוגמה מבודדת במפורש גם משחק ממחזור קודם ששוחק בעתיד; זו המחשת דליפה, לא השוואה של שני מדדים לאותה נקודת זמן.</p></div>

In [ ]:
postponed_context, postponed_tables = postponed_example(tables, features, official_rounds)
display(review_table(postponed_context))
display(review_table(postponed_tables))

<div dir="rtl"><h2>4. רשמי מול ספורטיבי — הורדות נקודות</h2><p><b>מה התא עושה:</b> מציג עונות שבהן יש הורדות או פער טכני, ומפריד בין הנקודות הספורטיביות, התוספת הטכנית לצורך האימות והנקודות הרשמיות. לצורך המודלים משתמשים רק בספורטיבי.</p>
<p>ב-<bdi dir="ltr">2011/2012</bdi> הורדות נקודות השפיעו על קו הירידה: לכן ״ירדה בפועל״ היא תווית אבחון של הטבלה הרשמית, ולא יעד אימון ספורטיבי. גם המיקום הספורטיבי כאן משתמש בשיבוץ הפלייאוף הידוע בפועל.</p></div>

In [ ]:
official_comparison = sporting_official_comparison(tables, rules, official_finals)
display(
    review_table(
        official_comparison.loc[
            official_comparison.rank_changed | official_comparison.deduction.ne(0)
        ]
    )
)

<div dir="rtl"><h2>השיבוץ ההיסטורי נשמר — הנקודות נשארות ספורטיביות</h2><p><b>מה התא עושה:</b> מציג את כל ההבדלים בין שיבוץ ספורטיבי מחושב לבין השיבוץ מהלוח בפועל, עם הסיבה המאומתת.</p>
<p>ב-<bdi dir="ltr">2010/2011</bdi> הניצחון הטכני של ב״ש על אשדוד נתן לה שלוש נקודות נוספות והכניס אותה לאמצעי. בלעדיו בית״ר הייתה באמצעי וב״ש בתחתון. בפועל בית״ר שיחקה חמישה משחקי תחתון במאבק ירידה וב״ש שלושה משחקי אמצעי. ב-M0 משתמשים בשיבוץ מהלוח כמידע K רק מרגע הפיצול; ב-M4 הפיצול ייגזר מהמסלול המדומה. final_stage נשאר לאימות בלבד.</p></div>

In [ ]:
display(review_table(split_comparison(tables, validation)))

<div dir="rtl"><h2>5. כמה משחקים נשארים בלי יעד תחרותי פתוח?</h2><p><b>מה התא עושה:</b> סופר בכל עונה ושלב משחקים שבהם לאף קבוצה, לאחת או לשתיהן אין יעד פתוח. המכנה הוא משחקים, ולא שתי שורות הקבוצה של כל משחק.</p>
<p><b>״אין על מה לשחק״ כאן הוא דגל מתמטי שמרני, לא ציון מוטיבציה:</b> הוא בודק אליפות, קו עליון וירידה, ובעליון ובעונות בלי פלייאוף גם את קווי המקומות 2, 3 ו-4 (ה-ת10). שוויון נקודות נשאר פתוח. לפני הפיצול מוכיחים פסילת אליפות דרך פסילה מהעליון, ובטיחות דרך הבטחת שיבוץ שאינו תחתון. אחרי הפיצול משתמשים במעטפת נקודות בתוך הקבוצה. השיטה עשויה לזהות הכרעה מאוחר; היא אינה מניחה שקבוצה בטוחה תפסיק להתאמץ.</p>
<p>באמצעי הישן אין יעד רלוונטי, ולכן כל הקבוצות בו מסומנות בלי יעד. בעליון הירידה כבר מוכרעת; בתחתון האליפות והקו העליון כבר מוכרעים.</p><p><b>למה נוספו קווי המקום (ה-ת10)?</b> גם אחרי שהאליפות הוכרעה, קבוצות בעליון עוד נלחמות על הסגנות ועל אירופה. לכן ״מוכרע״ מחייב שהצד של הקבוצה ביחס לכל קו בין המקומות <bdi dir="ltr">2|3, 3|4, 4|5</bdi> יהיה נעול במעטפת הנקודות. המקום הרביעי נכלל תמיד מטעמי שמרנות, בלי לנסות לקבוע כאן מי תקבל כרטיס אירופי; אירופה אינה יעד חיזוי נפרד.</p><p>טבלת ההשוואה בעליון מחושבת כאן מהעמודות בזיכרון: ההגדרה הישנה היא AND של title_decided, cut_decided ו-relegation_decided, והחדשה דורשת גם places_decided. שיעור שורות קבוצה אינו שיעור משחקים; המכנים מוצגים בנפרד.</p><p><b>מגבלה מתועדת — מבחן הירידה:</b> בעונות <bdi dir="ltr">2009/2010–2010/2011</bdi> מקום 14 הולך למבחן ירידה, ומקומות <bdi dir="ltr">15–16</bdi> יורדים ישירות. הדגל בתחתון רואה את שלושתם כצד אחד של קו הירידה. לכן קבוצה שכבר לא יכולה לסיים מעל מקום 14 מסומנת ״מוכרע״, אף שהיא עוד נלחמת על מבחן הירידה ולא על ירידה ישירה. המגבלה נוגעת רק לתחתון בשתי העונות; היא מתועדת כאן ללא שינוי בקוד הדגל.</p></div>

In [ ]:
dead_summary, match_flags = dead_game_summary(features)
display(review_table(dead_summary, percent=("at_least_one_share", "both_share")))
stage_totals = (
    dead_summary.groupby("stage")[["games", "neither", "one", "both"]]
    .sum()
    .reindex(["regular", "upper", "middle", "lower"])
    .reset_index()
)
display(review_table(stage_totals))

legacy_flag = features[["title_decided", "cut_decided", "relegation_decided"]].all(axis=1)
upper_comparison_rows = []
for label, flags in [("לפני ה-ת10", legacy_flag), ("אחרי ה-ת10", features.nothing_to_play)]:
    upper_flags = flags.loc[features.stage.eq("upper")]
    upper_comparison_rows.append(
        {
            "definition": label,
            "team_games": len(upper_flags),
            "dead_team_rows": int(upper_flags.sum()),
            "dead_team_share": float(upper_flags.mean()),
            "both": int(upper_flags.groupby(level="game_id").sum().eq(2).sum()),
        }
    )
upper_comparison = pd.DataFrame(upper_comparison_rows)
display(HTML('<div dir="rtl"><b>בעליון — הדגל לפני ואחרי ה-ת10</b></div>'))
display(review_table(upper_comparison, percent=("dead_team_share",)))
dead_counts = match_flags.dead_teams.value_counts().reindex([0, 1, 2], fill_value=0)
display(
    HTML(
        f'<div dir="rtl">מתוך {len(match_flags):,} משחקים, '
        f"ב-{int(dead_counts[1] + dead_counts[2]):,} "
        f"({100 * (dead_counts[1] + dead_counts[2]) / len(match_flags):.2f}%) "
        "לפחות קבוצה אחת בלי יעד פתוח לפי ההגדרה החדשה.</div>"
    )
)

<div dir="rtl"><h2>התפלגות המשחקים בלי יעד לאורך העונות</h2><p><b>מה התא עושה:</b> מציג את שיעור המשחקים עם לפחות קבוצה אחת בלי יעד פתוח, בנפרד לכל שלב. בריחוף מופיעים גם המכנים ומספר המשחקים שבהם קבוצה אחת או שתיים בלי יעד. זהו תיאור של הדגל שהוגדר, לא אומדן להשפעת מוטיבציה על תוצאות.</p></div>

In [ ]:
dead_figure = dead_games_chart(dead_summary)
display(dead_figure)

<div dir="rtl"><h2>דוגמאות: אלופה שמגיעה למשחק עם תואר מוכרע</h2><p><b>מה התא עושה:</b> לכל עונה מציג את המשחק הראשון של המובילה שבו הדגל כבר מוכיח שהאליפות שלה, וממיין לפי משחקים שנותרו בשלב.</p>
<p><b>זה אינו תאריך ההכרעה המתמטית המדויק:</b> האליפות יכולה להיות מוכרעת בין משחקי האלופה, והדגל השמרני עשוי לזהות זאת מאוחר. אלו דוגמאות מתועדות לפני משחק, ולא דירוג מוחלט של ״האליפות המוקדמת בהיסטוריה״.</p></div>

In [ ]:
display(review_table(championship_examples(tables, features)))

<div dir="rtl"><h2>6. משחקים עד סוף השלב — אחרי המשחק הנחזה</h2><p><b>מה התא עושה:</b> מציג את התפלגות המונה ומוודא שאינו שלילי. games_remaining_stage כולל את המשחק הנחזה; rounds_to_stage_end מפחית אותו. הספירה מחושבת מהפורמט פחות משחקי העבר, ולא ממספר שורות עתידיות.</p>
<p>משחק טכני שהוחרג אינו תוצאה ספורטיבית שנצפתה. לכן אצל קבוצה כזו המונה עשוי להשאיר משחק אחד נוסף גם אחרי משחקה האחרון בפועל. זה מרחיב את מעטפת האפשרויות ומאחר הכרעות; לא ממלאים תוצאה או מעניקים נקודות למשחק שהוצא.</p></div>

In [ ]:
stage_distribution = stage_end_distribution(features)
stage_figure = stage_end_chart(stage_distribution)
display(stage_figure)

<div dir="rtl"><h2>רישום הפיצ׳רים ומעמדם</h2><p><b>מה התא עושה:</b> מציג את הרשימה המפורשת שמותרת בפלט. עמודה לא רשומה, לרבות שופט, מאמן, אצטדיון או אירוע מתוך המשחק הנחזה, מכשילה את הבדיקה. לדגל nothing_to_play יש מעמד ברירת מחדל עבור M1 והנחת Elo, אך הוא עדיין מועמד לאבלציה ב-M2. עצם הרישום אינו הוכחה לתרומה לחיזוי.</p></div>

In [ ]:
display(review_table(registry_overview()))

<div dir="rtl"><h2>שמירה בשכבה הנגזרת</h2><p><b>מה התא עושה:</b> שומר גרסה 2 של טבלת הפיתוח מחוץ ל-data/clean, בלי לדרוס או למחוק את גרסה 1. המטא-דאטה מתארת את פערי ה-NA ואת הדגל לפי ה-ת10, עם גרסה, commit, hash של הקלט ושל הקובץ, כלל as_of וסוף אימון ריק משום שאין כאן רכיב נלמד. הגרפים נשמרים גם כ-HTML וגם כ-PNG. אין כאן פקודת CLI; היא תגיע בפעימה C.</p></div>

In [ ]:
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
artifact, metadata = save_table_state(
    features, input_sha256=input_fingerprint(tables, rules), commit=commit
)
figure_dir = Path("reports/figures/prematch")
figure_dir.mkdir(parents=True, exist_ok=True)
for name, figure in [("dead_games", dead_figure), ("stage_end", stage_figure)]:
    figure.write_html(figure_dir / f"{name}.html", include_plotlyjs=True)
    figure.write_image(figure_dir / f"{name}.png", width=1400, height=650, scale=1.5)
display(
    HTML(
        f'<div dir="rtl">נשמרו הטבלה והמטא-דאטה '
        f'ב-<bdi dir="ltr">{artifact.parent}</bdi>, '
        f'ושני הגרפים ב-<bdi dir="ltr">{figure_dir}</bdi>. '
        f"זמן ריצה עד כאן: {time.perf_counter() - started:.2f} שניות.</div>"
    )
)

<div dir="rtl"><h2>מה הושלם ומה ממשיך לפעימה B</h2><p>יש כעת חוזה ברור למצב הטבלה לפני משחק: תוצאות עבר בלבד, שיבוץ היסטורי ידוע, נקודות ספורטיביות, פיצ׳רים רשומים ודגלים עם הוכחה שמרנית. הטבלאות הרשמיות שימשו לבדיקת תקינות בלבד, עם סיווג מפורש של ארטיפקטים, משחקים טכניים, הורדות וסדר תצוגה.</p>
<p>זו תשובה ראשונה לשאלת המחקר 6א ב-VISION: כמה משחקים נשארים בלי אחד היעדים שהוגדרו. כדי לטעון להשפעה על הביצועים, צריך בהמשך קנה מידה של חוזק וציפייה לתוצאה; הדגל לבדו אינו מוכיח ירידת מוטיבציה.</p>
<p>פעימה B תוסיף כושר, התקפה והגנה בנפרד ו-Elo מכויל לפי התכנון המאושר. פעימה C תוסיף הקשר עונתי ושכבת prematch מאוחדת. לא מומשו כאן מודלים, כיול, חשיבות משחק או סימולציית עונה.</p><p><b>ממצא תיאורי:</b> מתוך 4,315 משחקי פיתוח, ב-156 משחקים (3.62%) קבוצה אחת בלי יעד פתוח וב-89 (2.06%) שתיהן; יחד 245 משחקים (5.68%). לדוגמה, מכבי ת״א הגיעה למשחק ב-7.4.2019 עם אליפות מוכחת בדגל, כשנותרו עוד שבעה משחקי שלב אחרי אותו משחק. זהו מועד תצפית לפני משחק שלה, ולא טענה שזה יום ההכרעה בפועל.</p></div>